In [8]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.linear_model import LinearRegression

# 1. Cargar datos
df = pd.read_excel('Motor Trend Car Road Tests.xlsx')
X = df[['hp', 'qsec']]
y = df['mpg']

# 2. Regresión
lr = LinearRegression()
lr.fit(X, y)

print("Intercepto:", lr.intercept_)
print("Betas hp y qsec:", lr.coef_)

# 2a. Intervalos de confianza
predicciones = lr.predict(X)
mse = np.sum((y - predicciones)**2) / (len(X) - 3)

# Armamos la matriz en fa para sacar el error estandar
X_mat = np.c_[np.ones(len(X)), X]
std_err = np.sqrt(np.diag(np.linalg.inv(X_mat.T @ X_mat) * mse))

t_val = stats.t.ppf(0.975, df=len(X)-3)
betas = np.append(lr.intercept_, lr.coef_)

print("\nIntervalos al 95%:")
for i, var in enumerate(['Intercepto', 'hp', 'qsec']):
    print(var, "->", betas[i] - t_val*std_err[i], "a", betas[i] + t_val*std_err[i])

# 3. Botsrap (10000 muestras
print("\n--- Resultados Botstrap ---")
betas_boot = []

for i in range(1000):
    muestra = df.sample(n=len(df), replace=True)
    lr_b = LinearRegression()
    lr_b.fit(muestra[['hp', 'qsec']], muestra['mpg'])
    betas_boot.append(np.append(lr_b.intercept_, lr_b.coef_))

df_boot = pd.DataFrame(betas_boot, columns=['Intercepto', 'hp', 'qsec'])

print("\nPromedios:")
print(df_boot.mean())

print("\nIntervalos Bo  tstrap (2.5% y 97.5%):")
print(df_boot.quantile([0.025, 0.975]))

Intercepto: 48.32370516913444
Betas hp y qsec: [-0.08459304 -0.88657962]

Intervalos al 95%:
Intercepto -> 25.614893944890643 a 71.03251639337824
hp -> -0.11308884885538409 a -0.056097238492801314
qsec -> -1.979929488063474 a 0.20677023879492984

--- Resultados Botstrap ---

Promedios:
Intercepto    50.082913
hp            -0.087838
qsec          -0.966526
dtype: float64

Intervalos Botstrap (2.5% y 97.5%):
       Intercepto        hp      qsec
0.025   30.769110 -0.119894 -2.103681
0.975   73.669354 -0.057570 -0.098119


## Paso 4: Comparación entre 2 y 3
Los coeficientes (Betas) y los intervalos de confianza de la regresión original salieron casi idénticos a los promedios que nos dio el Bootstrap. Esto demuestra que nuestro modelo es súper estable y que no hay valores raros (outliers) en los datos que estén afectando nuestra predicción.

### Paso 5: Búsqueda Aleatoria de Mejores Variables (Agregation)

In [2]:
import pandas as pd
import random
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

df = pd.read_excel('Motor Trend Car Road Tests.xlsx')

modelos = []
R2 = []
vars_usadas = []
models_test = []

columnas_disponibles = ['cyl', 'disp', 'hp', 'drat', 'wt', 'qsec', 'vs', 'am', 'gear', 'carb']

for i in range(1001):
    cols_al_azar = random.sample(columnas_disponibles, 3)
    X = df[cols_al_azar]
    y = df['mpg']

    X = sm.add_constant(X)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

    modelo_ols = sm.OLS(y_train, X_train)
    resultados = modelo_ols.fit()

    predicciones_test = resultados.predict(X_test)

    modelos.append(resultados)
    vars_usadas.append(cols_al_azar)
    models_test.append(predicciones_test)

    score = r2_score(y_test, predicciones_test)
    R2.append(score)

mejor_indice = R2.index(max(R2))
print(f"El mejor R^2 en Test fue: {R2[mejor_indice]:.4f}")
print(f"Las mejores 3 variables fueron: {vars_usadas[mejor_indice]}")

El mejor R^2 en Test fue: 0.7775
Las mejores 3 variables fueron: ['carb', 'am', 'disp']
